In [ ]:
#!/usr/bin/env python3
"""
Balanced Hybrid RL Intelligent Scheduler
 - 9 Edge Pods + Cloud
 - DQN + Heuristic Safety Layer
 - Reward Shaping for Correct Edge/Cloud Choice
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import pickle
from sklearn.preprocessing import MinMaxScaler

# ============================
# DEVICE
# ============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================
# CONFIG
# ============================
DATASET_PATH = "/kaggle/input/scheduler-dataser/scheduler_dataset.csv"

STATE_FEATURES = [
    "cpu_percent",
    "memory_percent",
    "queue_length",
    "task_size_bytes"
]

NUM_PODS = 9
NUM_ACTIONS = NUM_PODS + 1    # 9 pods + cloud

EPISODES = 220
BATCH_SIZE = 64
GAMMA = 0.95
LR = 1e-3
MEMORY_SIZE = 50000
TARGET_UPDATE_INTERVAL = 25

# Reward weights
LAMBDA_CPU = 0.4
LAMBDA_MEM = 0.3
LAMBDA_QUEUE = 0.2
LAMBDA_TASK = 0.1


# ============================
# LOAD DATA
# ============================
df = pd.read_csv(DATASET_PATH)

for c in STATE_FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna().reset_index(drop=True)

print("Dataset loaded:", df.shape)
print("Pods found:", sorted(df["pod_id"].unique()))

scaler = MinMaxScaler()
df[STATE_FEATURES] = scaler.fit_transform(df[STATE_FEATURES])


# ============================
# BASE REWARD
# ============================
def compute_reward(row):
    return -(
        LAMBDA_CPU * row["cpu_percent"]
        + LAMBDA_MEM * row["memory_percent"]
        + LAMBDA_QUEUE * row["queue_length"]
        + LAMBDA_TASK * row["task_size_bytes"]
    )

df["reward"] = df.apply(compute_reward, axis=1)
print("Reward range:", df["reward"].min(), "to", df["reward"].max())


# ============================
# DQN MODEL
# ============================
class DQN(nn.Module):
    def __init__(self, input_dim, actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, actions)
        )

    def forward(self, x):
        return self.net(x)


state_dim = len(STATE_FEATURES)

q_net = DQN(state_dim, NUM_ACTIONS).to(device)
target_net = DQN(state_dim, NUM_ACTIONS).to(device)
target_net.load_state_dict(q_net.state_dict())
target_net.eval()

optimizer = optim.Adam(q_net.parameters(), lr=LR)
loss_fn = nn.MSELoss()

memory = deque(maxlen=MEMORY_SIZE)


# ============================
# HEURISTIC (Balanced)
# ============================
CPU_OVERLOAD = 0.92
MEM_OVERLOAD = 0.95
QUEUE_OVERLOAD = 0.85

def heuristic_override(row):
    cpu = row["cpu_percent"]
    mem = row["memory_percent"]
    queue = row["queue_length"]
    task = row["task_size_bytes"]

    # strong overload → cloud
    if cpu > CPU_OVERLOAD or mem > MEM_OVERLOAD or queue > QUEUE_OVERLOAD:
        return NUM_ACTIONS - 1

    # large task + moderate pressure → cloud
    if task > 0.7 and (cpu > 0.75 or queue > 0.75):
        return NUM_ACTIONS - 1

    return None


# ============================
# EXPERIENCE
# ============================
def sample_experience():
    idx = np.random.randint(0, len(df) - 2)

    s = df.iloc[idx]
    s_next = df.iloc[idx + 1]

    state = torch.tensor(
        s[STATE_FEATURES].values.astype(np.float32),
        device=device
    )

    next_state = torch.tensor(
        s_next[STATE_FEATURES].values.astype(np.float32),
        device=device
    )

    reward = torch.tensor(s["reward"], dtype=torch.float32, device=device)

    with torch.no_grad():
        q_vals = q_net(state)
        dqn_action = torch.argmax(q_vals).item()

    override = heuristic_override(s)
    action = override if override is not None else dqn_action

    return state, action, reward, next_state, False


# ============================
# TRAINING STEP + REWARD SHAPING
# ============================
def train_step():
    if len(memory) < BATCH_SIZE:
        return None

    batch = random.sample(memory, BATCH_SIZE)

    states = torch.stack([b[0] for b in batch])
    actions = torch.tensor([b[1] for b in batch], device=device)
    rewards = torch.stack([b[2] for b in batch])
    next_states = torch.stack([b[3] for b in batch])

    # ---------- REWARD SHAPING ----------
    for i, a in enumerate(actions):
        # CLOUD penalty
        if a.item() == NUM_ACTIONS - 1:
            rewards[i] -= 0.4

        else:
            cpu = states[i][0].item()
            queue = states[i][2].item()

            # Healthy edge bonus
            if cpu < 0.6 and queue < 0.4:
                rewards[i] += 0.3
    # ------------------------------------

    q_vals = q_net(states)
    q_selected = q_vals.gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        q_next = target_net(next_states).max(dim=1)[0]
        targets = rewards + GAMMA * q_next

    loss = loss_fn(q_selected, targets)

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(q_net.parameters(), 1.0)
    optimizer.step()

    return loss.item()


# ============================
# TRAIN LOOP
# ============================
print("\n=========== TRAINING START 🚀 ===========\n")

for ep in range(EPISODES):
    total_reward = 0
    losses = []

    for _ in range(350):
        exp = sample_experience()
        memory.append(exp)
        total_reward += exp[2].item()

        loss = train_step()
        if loss:
            losses.append(loss)

    if ep % TARGET_UPDATE_INTERVAL == 0:
        target_net.load_state_dict(q_net.state_dict())

    print(f"Episode {ep+1}/{EPISODES} | Reward={total_reward:.2f} | Loss={np.mean(losses):.5f}")

# ============================
# SAVE
# ============================
torch.save(q_net.state_dict(), "hybrid_dqn_9pods_balanced.pt")

with open("state_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("\n=========== TRAINING COMPLETE 🎯 ===========")
print("Saved:")
print(" Model  -> hybrid_dqn.pt")
print(" Scaler -> state_scaler.pkl")


Using device: cuda
Dataset loaded: (6729, 20)
Pods found: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Reward range: -0.822068464068231 to -0.16612248420558917

=========== TRAINING START 🚀 ===========

Episode 1/220 | Reward=-165.97 | Loss=0.00808
Episode 2/220 | Reward=-169.34 | Loss=0.03805
Episode 3/220 | Reward=-165.80 | Loss=0.03446
Episode 4/220 | Reward=-164.20 | Loss=0.03452
Episode 5/220 | Reward=-165.95 | Loss=0.03460
Episode 6/220 | Reward=-163.98 | Loss=0.03543
Episode 7/220 | Reward=-168.60 | Loss=0.03524
Episode 8/220 | Reward=-165.06 | Loss=0.03549
Episode 9/220 | Reward=-162.92 | Loss=0.03505
Episode 10/220 | Reward=-165.39 | Loss=0.03474
Episode 11/220 | Reward=-163.03 | Loss=0.03462
Episode 12/220 | Reward=-160.83 | Loss=0.03472
Episode 13/220 | Reward=-165.83 | Loss=0.03430
Episode 14/220 | Reward=-164.73 | Loss=0.03417
Episode 15/220 | Reward=-164.60 | Loss=0.03393
Episode 16/220 | Reward=-167.71 | Loss=0.03341
Episode 17/220 | Reward=-166.60 | Loss=0.03397
Episode 18/220 | Reward=